# Stored chemical coverage before DockingMT preparation

Consumer evidence for DockingMT #5/#33 and MolSysMT #217. The provider audits
stored fields without repairing chemistry. Presence is distinct from scientific
validity; DockingMT's existing provisional-preparation safeguard remains active.
See `chemical_readiness_consumption.md` for contracts, axes and limits.


In [1]:
import hashlib
import json
from pathlib import Path
import sys

import molsysmt as msm
from dockingmt.preparation import prepare_ligand

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "devguide/validation/data/readiness/preparation_profile.json").is_file())
print(json.dumps({"python": sys.version.split()[0], "interpreter": sys.executable,
                  "provider_import": msm.__file__,
                  "qualified_provider_source": "3edbf8ad0a13b9a56a009c0bd3f707e54b807351",
                  "provider_readiness_sha256": hashlib.sha256(
                      (Path(msm.__file__).parent / "_private/chemical_readiness.py").read_bytes()).hexdigest()}, indent=2))

{
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_import": "/tmp/dockingmt-readiness-provider-3edbf8ad0/molsysmt/__init__.py",
  "qualified_provider_source": "3edbf8ad0a13b9a56a009c0bd3f707e54b807351",
  "provider_readiness_sha256": "62b715f2dc63dbee6e4bfd79437b1ed61e82e144e80ef35c88b4878bc9747540"
}


## Present chemical fields do not certify AutoDock assignments

Caffeine's SDF stores formal charges and complete chemical connectivity. Those
facts do not supply a named partial-charge model or qualified AutoDock typing.
The prepared metadata audits the selected source before hydrogen projection.


In [2]:
source = msm.convert(msm.systems["caffeine"]["caffeine.sdf"], to_form="molsysmt.MolSys")
prepared = prepare_ligand(source, selection="all")
summary = prepared.metadata["source_chemistry"]["chemical_readiness"]
assert summary["fields"]["formal_charge"]["status"] == "present"
assert summary["connectivity"]["declared_completeness"] == "complete"
assert prepared.metadata["charge_source"] == "zero_placeholder"
assert "heuristic" in prepared.metadata["atom_type_source"]
assert "docking_readiness" in summary["unassessed_checks"]
assert "ready" not in summary
print(json.dumps({"source_atoms": summary["n_atoms"], "retained_atoms": prepared.n_atoms,
                  "stored_fields": summary["fields"],
                  "explicit_H_atoms": summary["n_explicit_hydrogens"],
                  "charge_source": prepared.metadata["charge_source"],
                  "atom_type_source": prepared.metadata["atom_type_source"],
                  "unassessed_checks": summary["unassessed_checks"]}, indent=2))

{
  "source_atoms": 24,
  "retained_atoms": 14,
  "stored_fields": {
    "atom_type": {
      "status": "present",
      "n_assessed": 24,
      "n_present": 24,
      "n_missing": 0,
      "n_unsupported": 0,
      "n_conflict": 0,
      "origin_counts": {
        "unassessed": 24
      }
    },
    "atom_id": {
      "status": "present",
      "n_assessed": 24,
      "n_present": 24,
      "n_missing": 0,
      "n_unsupported": 0,
      "n_conflict": 0,
      "origin_counts": {
        "unassessed": 24
      }
    },
    "isotope": {
      "status": "missing",
      "n_assessed": 24,
      "n_present": 0,
      "n_missing": 24,
      "n_unsupported": 0,
      "n_conflict": 0,
      "origin_counts": {
        "unassessed": 24
      }
    },
    "formal_charge": {
      "status": "present",
      "n_assessed": 24,
      "n_present": 24,
      "n_missing": 0,
      "n_unsupported": 0,
      "n_conflict": 0,
      "origin_counts": {
        "unassessed": 24
      },
      "unit": "elemen

## Inspect original PDBQT through the provider

Prepared files retain supplied assignments and trees. The audit reports partial
connectivity and missing formal charges; it does not replace the PDBQT parser or
certify the supplied partial charges. Each original file remains unchanged.


In [3]:
rows = []
for name in ("1iep_ligand", "1s63_ligand", "5x72_ligand_p59", "5x72_ligand_p69"):
    path = root / "tests/data/vina_torsions" / (name + ".pdbqt")
    original = path.read_bytes()
    report = msm.physchem.get_chemical_readiness(path, chemical_state="structure", structure_indices=0)
    assert path.read_bytes() == original
    assert report["connectivity"]["declared_completeness"] == "partial"
    assert report["fields"]["formal_charge"]["status"] == "missing"
    rows.append({"case": name, "atoms": report["n_atoms"],
                 "connectivity": report["connectivity"]["declared_completeness"],
                 "formal_charge": report["fields"]["formal_charge"]["status"],
                 "coordinates": report["fields"]["coordinates"]["status"],
                 "explicit_H_atoms": len(report["explicit_hydrogen_atom_indices"]),
                 "unassessed_checks": report["unassessed_checks"]})
print(json.dumps(rows, indent=2))

[
  {
    "case": "1iep_ligand",
    "atoms": 40,
    "connectivity": "partial",
    "formal_charge": "missing",
    "coordinates": "present",
    "explicit_H_atoms": 3,
    "unassessed_checks": [
      "valence",
      "protonation",
      "missing_hydrogen_inventory",
      "aromaticity_perception",
      "stereogenicity",
      "periodic_geometry",
      "conformer_quality",
      "charge_model",
      "docking_readiness",
      "property_origin"
    ]
  },
  {
    "case": "1s63_ligand",
    "atoms": 30,
    "connectivity": "partial",
    "formal_charge": "missing",
    "coordinates": "present",
    "explicit_H_atoms": 1,
    "unassessed_checks": [
      "valence",
      "protonation",
      "missing_hydrogen_inventory",
      "aromaticity_perception",
      "stereogenicity",
      "periodic_geometry",
      "conformer_quality",
      "charge_model",
      "docking_readiness",
      "property_origin"
    ]
  },
  {
    "case": "5x72_ligand_p59",
    "atoms": 25,
    "connectivity": 

## Captured preparation overhead

Five warm-cache samples per role and profile, after one warm-up. The benchmark
substitutes only the prior `chemistry_evidence` operation from commit `559f7c3`;
the source, selection and remaining preparation code are shared. These local
samples are not a universal speed or memory claim. Reproduce using
`devtools/profile_preparation_readiness.py` and the qualified provider source.


In [4]:
profile = json.loads((root / "devguide/validation/data/readiness/preparation_profile.json").read_text())
print(json.dumps(profile, indent=2))

{
  "evidence": "Local warm-cache preparation samples, not a universal speed claim.",
  "baseline_source": "559f7c33096d80983f44de4df6a0263f5f4ff7ca",
  "baseline_operation": "Prior chemistry_evidence only; remaining preparation code unchanged.",
  "sampling": "Five paired rounds per role, alternating profile order after one warm-up per profile.",
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_import": "/tmp/dockingmt-readiness-provider-3edbf8ad0/molsysmt/__init__.py",
  "qualified_provider_source": "3edbf8ad0a13b9a56a009c0bd3f707e54b807351",
  "provider_readiness_sha256": "62b715f2dc63dbee6e4bfd79437b1ed61e82e144e80ef35c88b4878bc9747540",
  "cases": [
    {
      "role": "ligand",
      "profile": "legacy",
      "retained_atoms": 6,
      "median_seconds": 0.14574280753731728,
      "samples_seconds": [
        0.13853552285581827,
        0.1515391431748867,
        0.14417296182364225,
        0.15103231463